# 03 · Model evaluation

**Question:** standing at matchweek *k*, how well does the Poisson + Monte Carlo model call the
rest of the season, and does it beat simple baselines?

**Method:** walk-forward backtest on the complete `fermedo_2024_25` season. For each cut-off *k*,
team strength and venue factors are fitted on matchweeks ≤ *k* only, and the model is scored on
matchweeks > *k*, which it never saw.

**Model**

$$\lambda_{home} = \frac{att_{home}}{def_{away}}\,\bar g\,h \qquad \lambda_{away} = \frac{att_{away}}{def_{home}}\,\bar g\,a$$

with $\bar g$ the league goals per team per game. Three variants:

| model | home *h* | away *a* |
|---|---|---|
| `poisson_fixed_ha` | 1.15 | 1.00 (the original model) |
| `poisson_venue_fit` | fitted | fitted (moves goals between venues, keeps the total) |
| `poisson_no_ha` | 1.00 | 1.00 |

**Baselines:** `uniform` (⅓ each) and `base_rates` (training H/D/A frequencies) for single matches;
`ppg` (points per game so far × games in the season) for the final table.

**Scores:** RPS (ranked probability score, the standard for ordered H/D/A outcomes), Brier, log
loss and hit rate per match, all lower-is-better except hit rate; points MAE, Spearman ρ and the
probability given to the real champion for the table.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))  # works without `pip install -e .`

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

import numpy as np
import matplotlib.pyplot as plt

from season_model.evaluation import (
    DEFAULT_MODELS, PROB_COLUMNS, backtest, calibration_table, draw_rate_check,
    fit_venue_factors, ranked_probability_score,
)
from season_model.league_table import compute_league_table
from season_model.pipeline import load_season, validate_season
from season_model.predictor import simulate_season
from season_model.team_strength import compute_team_strength

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"poisson_fixed_ha": "#4C78A8", "poisson_venue_fit": "#F58518", "poisson_no_ha": "#54A24B",
          "uniform": "0.6", "base_rates": "0.35", "ppg": "0.35"}

EVAL_SEASON = "fermedo_2024_25"
N_SIMULATIONS = 2000
SEED = 42

played, remaining = load_season(EVAL_SEASON)
assert remaining.empty, "Backtests need a finished season"
n_weeks = int(played["Numeric Matchweek"].max())
CUTOFFS = list(range(3, n_weeks - 1, 2))
print(f"{EVAL_SEASON}: {len(played)} matches over {n_weeks} matchweeks; cut-offs {CUTOFFS}")

In [ ]:
result = backtest(played, CUTOFFS, models=DEFAULT_MODELS, n_simulations=N_SIMULATIONS, seed=SEED)
result.summary().round(3)

## 1. Single matches: does the model beat the baselines, and from when?

In [ ]:
ms = result.match_scores
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for model, g in ms.groupby("model"):
    ls = "--" if model in ("uniform", "base_rates") else "-"
    axes[0].plot(g["cutoff"], g["rps"], ls, marker="o", ms=3, color=COLORS[model], label=model)
    axes[1].plot(g["cutoff"], g["accuracy"], ls, marker="o", ms=3, color=COLORS[model], label=model)
axes[0].set(xlabel="cut-off matchweek", ylabel="RPS (lower is better)", title="Ranked probability score on the remaining matches")
axes[1].set(xlabel="cut-off matchweek", ylabel="hit rate", title="Most likely outcome correct")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Late cut-offs score few matches (8 per matchweek), so their points are noisy.
ms.pivot_table(index="cutoff", columns="model", values="rps").round(3).join(
    ms.groupby("cutoff")["n_matches"].first())

## 2. Is the difference between the Poisson variants real?

The three variants differ by a few thousandths of RPS. A paired bootstrap over matches at one
mid-season cut-off says whether that is more than resampling noise. Matches at different
cut-offs overlap, so they are not pooled.

In [ ]:
def paired_bootstrap(preds, a, b, cutoff, n_boot=5000, seed=0):
    sub = preds[preds["cutoff"] == cutoff]
    pa = sub[sub["model"] == a].sort_values(["Numeric Matchweek", "Home Team"])
    pb = sub[sub["model"] == b].sort_values(["Numeric Matchweek", "Home Team"])
    diff = (ranked_probability_score(pa[PROB_COLUMNS].to_numpy(), pa["outcome"].to_numpy())
            - ranked_probability_score(pb[PROB_COLUMNS].to_numpy(), pb["outcome"].to_numpy()))
    rng = np.random.default_rng(seed)
    boots = rng.choice(diff, size=(n_boot, len(diff)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"a": a, "b": b, "cutoff": cutoff, "n": len(diff), "mean RPS(a) - RPS(b)": diff.mean(),
            "95% CI low": lo, "95% CI high": hi, "CI excludes 0": not (lo <= 0 <= hi)}

MID = CUTOFFS[len(CUTOFFS) // 2]
pd.DataFrame([
    paired_bootstrap(result.predictions, "poisson_venue_fit", "poisson_fixed_ha", MID),
    paired_bootstrap(result.predictions, "poisson_no_ha", "poisson_fixed_ha", MID),
]).round(4)

## 3. Calibration: when the model says 60 %, does it happen 60 % of the time?

In [ ]:
cal = calibration_table(result.predictions[result.predictions["model"] == "poisson_fixed_ha"], n_bins=5)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], color="0.7", lw=1)
for outcome, g in cal.groupby("outcome"):
    ax.plot(g["mean_predicted"], g["observed"], "o-", label=outcome)
    for _, r in g.iterrows():
        ax.annotate(int(r["n"]), (r["mean_predicted"], r["observed"]), fontsize=7, xytext=(3, -8), textcoords="offset points")
ax.set(xlabel="predicted probability", ylabel="observed frequency", xlim=(0, 1), ylim=(0, 1),
       title="Calibration, poisson_fixed_ha, all cut-offs\n(labels = matches per bin)")
ax.legend(frameon=False)
plt.show()
cal.round(3)

In [ ]:
# Draws are the usual weak spot of independent Poisson goals. Check this league's direction.
strength_full = compute_team_strength(played)
pd.DataFrame({
    "fixed 1.15": draw_rate_check(played, strength_full, 1.15),
    "venue fit": draw_rate_check(played, strength_full, *fit_venue_factors(played)),
}).round(3)

## 4. The final table: does simulating beat carrying points-per-game forward?

In [ ]:
ts = result.table_scores
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for model, g in ts.groupby("model"):
    ls = "--" if model == "ppg" else "-"
    axes[0].plot(g["cutoff"], g["points_mae"], ls, marker="o", ms=3, color=COLORS[model], label=model)
    if model != "ppg":
        axes[1].plot(g["cutoff"], g["p_actual_champion"], marker="o", ms=3, color=COLORS[model], label=model)
axes[0].set(xlabel="cut-off matchweek", ylabel="mean |predicted - final points|", title="Final points error per team")
axes[1].set(xlabel="cut-off matchweek", ylabel="P(real champion wins title)", ylim=(0, 1),
            title=f"Probability given to the real champion ({compute_league_table(played).iloc[0]['Team']})")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
ts.pivot_table(index="cutoff", columns="model", values=["points_mae", "spearman"]).round(2)

## 5. Findings

The cell below prints the headline numbers from this run, so the text never goes stale when the
data or the model changes. Things to keep in mind when reading it:

* **One season, 16 teams.** Every number here comes from 240 matches. Differences of a few
  thousandths of RPS between the Poisson variants are inside the bootstrap noise of section 2;
  only the gap to the baselines is a finding.
* **Overdispersion.** Goals vary about twice as much as Poisson allows (notebook 02). A negative
  binomial or a Dixon–Coles draw correction are the obvious next models to test against these
  same baselines.
* **Strength never decays.** Matchweek 1 counts as much as last week. A time-weighted strength is
  another candidate.

In [ ]:
summary = result.summary()
best = summary.drop(index=["uniform", "base_rates", "ppg"], errors="ignore")["rps"].idxmin()
gain = 1 - summary.loc[best, "rps"] / summary.loc["uniform", "rps"]
mae_model = ts[ts["model"] == "poisson_fixed_ha"]["points_mae"].mean()
mae_ppg = ts[ts["model"] == "ppg"]["points_mae"].mean()
wins = (ts.pivot_table(index="cutoff", columns="model", values="points_mae")
          .pipe(lambda t: (t["poisson_fixed_ha"] < t["ppg"]).mean()))
print(f"Best match model by mean RPS: {best} ({summary.loc[best, 'rps']:.3f}), "
      f"{gain:.0%} better than uniform ({summary.loc['uniform', 'rps']:.3f}).")
print(f"Final-table points MAE: model {mae_model:.2f} vs ppg {mae_ppg:.2f}; "
      f"model better at {wins:.0%} of cut-offs.")

## 6. Apply it to the current season

The model with the evaluation above behind it, run on `mansores_2025_26`. **Read the validation
first**: this season holds Mansores' fixtures only, so every other team's strength rests on a
single match and their rows are close to noise. Re-scrape the full league in notebook 01 before
trusting anything except Mansores' own row.

In [ ]:
CURRENT = "mansores_2025_26"
MODEL_HOME, MODEL_AWAY = 1.15, 1.0   # swap for fit_venue_factors(c_played) to use the fitted variant

c_played, c_remaining = load_season(CURRENT)
display(validate_season(CURRENT, c_played, c_remaining)[["check", "ok", "value"]])

pred = simulate_season(c_played, c_remaining, n_simulations=5000, seed=SEED,
                       home_advantage=MODEL_HOME, away_factor=MODEL_AWAY)
pred.predicted_table

In [ ]:
pred.position_probabilities

In [ ]:
team = "mansores"
row = pred.position_matrix.loc[team]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(row.index, row.values, color="#4C78A8")
ax.set(xlabel="final position", ylabel="%", title=f"{team}: finishing position over {pred.n_simulations} simulations")
ax.set_xticks(row.index)
plt.show()